# 第 26 节：RAG 知识库与智能体记忆 —— 让系统"有知识、有记忆、越用越专业"

> 前置课程：第 24 节（多智能体 Plan-and-Execute + ReAct 执行）、第 25 节（A2A 审核与反馈循环）
> 前置知识：RAG 的原理与工程实现已在《RAG 课程包（10-15 节）》系统讲过 —— 本节只做**简短回顾**，
>           重点讲"怎么把现成的 RAG 能力接进多智能体系统"，并新增**短期记忆 / 长期记忆**两大能力
> 对应项目：SmartBrief（`smartbrief/`）
> 对应规格：`多智能体协作开发PRD（拆分）.md —— 子文档 5：第 26 节 RAG 知识库与智能体记忆深度融合`

---

## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 课程导入：产品经理的三个新痛点 | — |
| **Part 2** | 环境准备与项目结构 | `requirements.txt` / `.env.example` |
| **Part 3** | 先立框架：智能体的"记忆"分几种 | — |
| **Part 4** | 给 Writer 一本"专业词典"：RAG 术语注入（原理简短回顾） | `rag/*` + `agents/react_writer.py` |
| **Part 5** | 短期记忆：Checkpointer —— 让任务"断电续跑" | `memory/checkpointer.py` |
| **Part 6** | 长期记忆：把成功经验写进向量库 | `memory/long_term_memory.py` |
| **Part 7** | Planner 的经验复用：相似报告 Few-shot | `agents/planner.py` + `graph/state.py` |
| **Part 8** | 把记忆接进图：workflow 与 main 的改动 | `graph/workflow.py` + `main.py` |
| **Part 9** | API 与前端闭环 | `api/routes.py` + `frontend/src/views/` |
| **Part 10** | 端到端演示：记忆闭环完整跑一遍（模拟 LLM，无需 API Key） | 多模块联合 |

---

> **面向学员**：和前两节一样，本教程不是"背代码"，而是带你走一遍
> **真实项目迭代中遇到的问题 → 为什么这样设计 → 最终代码长什么样**。
> 第 24 节让系统"会分工"、第 25 节让系统"会把关"，第 26 节让系统**"有知识、有记忆"**。
> 建议按顺序逐 Cell 运行，观察每一次输出。


---
## Part 1：课程导入 —— 产品经理的三个新痛点
---

### 1.1 产品经理又来了

你交付了第 25 节的"会审稿"版本。产品经理试用一周后，带着三个反馈来找你：

> **反馈一（用词不专业）**："报告里把『同比』说成『同月比』、把『渗透率』说成『普及速度』……
> 我们明明维护着一份**行业术语表**，还沉淀了 5 份**历史报告模板**，为什么写手不用？"

> **反馈二（重启白干）**："报告写到一半，后端**一重启就从头再来**。
> 一个要跑 20 分钟的任务，断了就全白费？"

> **反馈三（没有积累）**："三个月前刚做完『智能手表』的报告，这次再做『智能手表 2026 趋势』，
> 规划结构和上次完全不一样，上次验证过的好框架（章节怎么排、先研究什么）又从头摸索一遍。"

把三个反馈放在一起看，其实是同一个大主题：

> **系统没有"知识"，也没有"记忆"** —— 可用的资料不会主动调用，做完的任务不留下经验。

> 🤔 **暂停思考**：如果让你给系统加上"知识和记忆"，你的第一版方案会是什么？
> 把术语表全文塞进每个 Agent 的 Prompt？把历史报告全文贴给 Planner？先把你的方案写下来，
> 再看下面的 Cell 0 —— 这三个"直觉方案"为什么都行不通。


In [1]:
# ============================================================
# Cell 0：v0 —— 三种"拍脑袋"解法，为什么都行不通？
# ============================================================

print("=" * 55)
print("  方案A：把术语表 / 模板全文贴进 Writer 的 Prompt")
print("=" * 55)
print("  伪代码：")
print("    SYSTEM_PROMPT += open('data/terminologies.txt').read()  # 30 条术语")
print("    SYSTEM_PROMPT += '\\n'.join(模板文件)                    # 5 份模板")
print()
print("  [问题]")
print("    1. 术语/模板会越来越多 → Prompt 越来越长 → 又贵又容易截断")
print("    2. 写一章只需 2~3 条相关术语，塞 30 条全是噪音")
print("    3. 运行期上传的新文档无法动态生效（写死在 Prompt 里）")
print()
print("=" * 55)
print("  方案B：把历史报告全文贴给 Planner 参考")
print("=" * 55)
print("  伪代码：")
print("    user = f'主题：{topic}\\n\\n历史报告全文：{history_report}'")
print()
print("  [问题]")
print("    1. 一份报告几千字，多存几份就把上下文塞爆")
print("    2. Planner 真正需要的是'章节结构'，不是正文全文")
print("    3. 没有'相似度'概念：无论主题多不相关都会硬塞，反而误导")
print()
print("=" * 55)
print("  方案C：自己在业务库里'存断点'")
print("=" * 55)
print("  伪代码：")
print("    while step < len(plan):")
print("        ...")
print("        save_progress(step, 部分草稿)   # 自己记录'跑到第几步'")
print()
print("  [问题]")
print("    1. 图里有循环（第 25 节审核回路），'跑到第几步'根本说不清")
print("    2. 图每执行一个节点，白板（state）都会变 —— 该存哪个版本？")
print("    3. 这是 LangGraph 框架的职责，自己造轮子必然有漏洞")
print()
print("  [结论] 三个方案本质都是'把记忆当字符串硬塞'。")
print("         第 26 节的正解是'检索 + 框架记忆'三件套：")
print("           ① RAG 知识库     —— 需要哪段知识，检索哪段（反馈一）")
print("           ② Checkpointer   —— 图状态自动存快照，断点续跑（反馈二）")
print("           ③ 长期记忆       —— 成功经验入库，相似任务 Few-shot（反馈三）")


  方案A：把术语表 / 模板全文贴进 Writer 的 Prompt
  伪代码：
    SYSTEM_PROMPT += open('data/terminologies.txt').read()  # 30 条术语
    SYSTEM_PROMPT += '\n'.join(模板文件)                    # 5 份模板

  [问题]
    1. 术语/模板会越来越多 → Prompt 越来越长 → 又贵又容易截断
    2. 写一章只需 2~3 条相关术语，塞 30 条全是噪音
    3. 运行期上传的新文档无法动态生效（写死在 Prompt 里）

  方案B：把历史报告全文贴给 Planner 参考
  伪代码：
    user = f'主题：{topic}\n\n历史报告全文：{history_report}'

  [问题]
    1. 一份报告几千字，多存几份就把上下文塞爆
    2. Planner 真正需要的是'章节结构'，不是正文全文
    3. 没有'相似度'概念：无论主题多不相关都会硬塞，反而误导

  方案C：自己在业务库里'存断点'
  伪代码：
    while step < len(plan):
        ...
        save_progress(step, 部分草稿)   # 自己记录'跑到第几步'

  [问题]
    1. 图里有循环（第 25 节审核回路），'跑到第几步'根本说不清
    2. 图每执行一个节点，白板（state）都会变 —— 该存哪个版本？
    3. 这是 LangGraph 框架的职责，自己造轮子必然有漏洞

  [结论] 三个方案本质都是'把记忆当字符串硬塞'。
         第 26 节的正解是'检索 + 框架记忆'三件套：
           ① RAG 知识库     —— 需要哪段知识，检索哪段（反馈一）
           ② Checkpointer   —— 图状态自动存快照，断点续跑（反馈二）
           ③ 长期记忆       —— 成功经验入库，相似任务 Few-shot（反馈三）


### 1.2 拆解：人是怎么"越用越专业"的？

一个干了三年的分析师 vs 一个刚入职的新人，差别不在"更聪明"，而在三点：

| 人的能力 | 本质 | 对应到系统 |
|---------|------|-----------|
| 查工具书/资料 | **用的时候才检索**，而不是把整本书背下来 | RAG 知识库（向量检索 + 按需注入） |
| 手头任务干到一半 | 脑子里**记着做到哪了**，接个电话回来继续 | Checkpointer（图状态快照 / 断点续跑） |
| 上次项目的经验 | 记得**结构和套路**，下次直接套框架 | 长期记忆（报告入库 + 相似检索 Few-shot） |

> ⚠️ 一个关键认知：**大模型本身是"无状态"的** —— 同样的 Prompt 丢给它，
> 它不记得上次写过什么、你给它喂过什么资料。所以"记忆"必须由**工程层**提供，
> 这就是第 26 节的全部内容。注意区分：
> - "Prompt 缓存 / 对话历史" 是**本次调用内**的上下文（工作记忆）；
> - 第 26 节要做的是 **跨调用、跨任务、甚至跨重启** 的持久记忆。

### 1.3 本节课的"开发旅程"

```
痛点 → 试错 → 最终设计 → 项目里的最终代码 → 端到端演示
```

| 步骤 | 我们解决什么 | 项目里的实现 |
|------|-------------|-------------|
| ① | Writer 用词不专业 | RAG 术语表检索 → 注入 Prompt（Part 4） |
| ② | 任务跑一半重启就白干 | Checkpointer 短期记忆（Part 5） |
| ③ | 同主题第二次还是从零开始 | 长期记忆 + Planner Few-shot（Part 6 / 7） |
| ④ | 把①②③串进图 / 启动 / API | workflow / main / routes（Part 8 / 9） |
| ⑤ | 前端可视化闭环 | 知识库页 + 记忆命中卡片 + 恢复执行（Part 9） |

> 📚 **关于 RAG 原理**：嵌入（Embedding）、分块（Chunking）、余弦相似度、召回策略……
> 这些在第 10-15 节的 RAG 课程包已系统讲透：
> `资料/.../02-大模型全栈开发实战/10-15/slides/01-多格式文档加载与数据清洗.ipynb`
> `~ 05-RAG系统评测、优化与部署.ipynb`。
> 本节遇到 RAG 概念时只给"一句结论 + 回看指引"，不再重复推导。


---
## Part 2：环境准备与项目结构
---

### 2.1 新增依赖与配置

第 26 节给后端新增两类能力：

**（1）RAG 向量知识库**（复用第 10-15 节的技术栈：ChromaDB + bge-m3）

| 依赖 | 用途 |
|------|------|
| `chromadb>=0.5.0` | 本地向量数据库（`rag_collection` + `memory_collection` 两个集合） |
| `python-docx` / `PyPDF2` / `openpyxl` | 知识库上传 .docx / .pdf / .xlsx 文档 |
| 嵌入模型 API | SiliconFlow `BAAI/bge-m3`（1024 维、中文优化，兼容 OpenAI Embeddings 接口） |

**（2）短期记忆（Checkpointer）**

| 依赖 | 用途 |
|------|------|
| `langgraph-checkpoint-sqlite>=1.0.0` | 把"图运行时状态"快照持久化到 `backend/checkpoints.db` |
| `aiosqlite>=0.20.0` | SQLite 异步连接（AsyncSqliteSaver 的运行依赖） |

`.env` 新增三行**嵌入模型**配置（复制 `.env.example` 并填 Key 即可，结构跟 `OPENAI_*` 完全平行）：

```bash
# ===== 嵌入模型配置（第 26 节 RAG，复用 SiliconFlow 兼容 OpenAI Embeddings 接口）=====
EMBEDDING_BASE_URL=https://api.siliconflow.cn/v1
EMBEDDING_API_KEY=sk-xxxx
EMBEDDING_MODEL=BAAI/bge-m3
```

> 为什么是"第 26 节才引入"？—— 本节的 RAG / 记忆都依赖**语义相似度检索**：
> 术语"YoY"和句子"同比增长率该怎么说"必须能被**向量距离**拉近，关键词匹配做不到。


In [2]:
# ============================================================
# Cell 1：检查环境 + 把 backend 加入 sys.path
# ============================================================
import os
import sys

print("=" * 55)
print("  环境检查（第 26 节）")
print("=" * 55)
print(f"  Python 版本: {sys.version.split()[0]}")

# 兼容两种运行位置：
#   1) notebook 在 proj05 根目录运行时 -> backend = ./smartbrief/backend
#   2) notebook 在 smartbrief 目录运行时 -> backend = ./backend
BACKEND = os.path.join(os.getcwd(), "smartbrief", "backend")
if not os.path.isdir(BACKEND):
    BACKEND = os.path.join(os.getcwd(), "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)
print(f"  backend 路径: {BACKEND}")

for name in ("langgraph", "chromadb"):
    try:
        mod = __import__(name, fromlist=["*"])
        print(f"  [OK] {name} 版本: {getattr(mod, '__version__', '已安装')}")
    except Exception as e:
        print(f"  [警告] 缺少 {name}: {e}")
        print("         请先安装: pip install 'langgraph>=1.0' 'chromadb>=0.5.0'")

# 检查 .env 是否配置了嵌入 Key（本 Notebook 演示全离线，真实 RAG 才需要）
env_file = os.path.join(BACKEND, ".env")
key = ""
if os.path.exists(env_file):
    for line in open(env_file, encoding="utf-8"):
        if line.startswith("EMBEDDING_API_KEY="):
            key = line.split("=", 1)[1].strip()
if key:
    print(f"  [OK] EMBEDDING_API_KEY 已配置: {key[:6]}****")
else:
    print("  [提示] .env 未配置 EMBEDDING_API_KEY（不影响本 Notebook 的离线演示）")
    print("         想体验真实 RAG：复制 backend/.env.example 为 .env 并填写 Key")


  环境检查（第 26 节）
  Python 版本: 3.10.20
  backend 路径: c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj05\smartbrief\backend
  [OK] langgraph 版本: 已安装
  [OK] chromadb 版本: 1.5.9
  [OK] EMBEDDING_API_KEY 已配置: sk-lae****


In [3]:
# ============================================================
# Cell 2：看一眼新增的依赖、配置与预置数据
# ============================================================

def show_file(path, title, max_lines=None):
    print("=" * 55)
    print(f"  {title}  ({os.path.relpath(path, BACKEND)})")
    print("=" * 55)
    lines = open(path, encoding="utf-8").read().splitlines()
    if max_lines:
        lines = lines[:max_lines]
    for no, line in enumerate(lines, start=1):
        print(f"{no:>4} | {line}")
    print()

# 1) requirements.txt 里第 26 节相关条目
print("=" * 55)
print("  requirements.txt（第 26 节相关条目）")
print("=" * 55)
for no, line in enumerate(
    open(os.path.join(BACKEND, "requirements.txt"), encoding="utf-8").read().splitlines(),
    start=1,
):
    if any(k in line for k in ("chromadb", "python-docx", "PyPDF2", "openpyxl",
                               "checkpoint", "aiosqlite")):
        print(f"{no:>4} | {line}")
print()

# 2) .env.example 的嵌入配置段
show_file(os.path.join(BACKEND, ".env.example"), ".env.example 嵌入配置段")

# 3) 术语表前 6 条（RAG 预置数据，后面 Writer 注入的就是它）
show_file(os.path.join(BACKEND, "data", "terminologies.txt"),
          "data/terminologies.txt（30 条行业术语，前 6 条）", max_lines=6)


  requirements.txt（第 26 节相关条目）
  18 | chromadb>=0.5.0              # 向量数据库（rag_collection / memory_collection）
  19 | python-docx>=1.0.0           # .docx 文档加载（RAG 上传）
  20 | PyPDF2>=3.0.0                # .pdf 文档加载（RAG 上传）
  21 | openpyxl>=3.1.0              # .xlsx 文档加载（RAG 上传）
  22 | langgraph-checkpoint-sqlite>=1.0.0   # 短期记忆 Checkpointer（checkpoints.db；v3.x 的 from_conn_string 需 async with 打开连接）
  23 | aiosqlite>=0.20.0            # SQLite 异步连接（langgraph-checkpoint-sqlite 运行时依赖）

  .env.example 嵌入配置段  (.env.example)
   1 | # SmartBrief 后端配置示例：复制为 .env 并填入真实值
   2 | # 支持任意 OpenAI 兼容接口（DeepSeek / Kimi / Ollama 等），只需改 BASE_URL 与 MODEL
   3 | 
   4 | OPENAI_BASE_URL=https://api.openai.com/v1
   5 | OPENAI_API_KEY=sk-xxxx
   6 | OPENAI_MODEL=gpt-4o-mini
   7 | 
   8 | # ===== 嵌入模型配置（第 26 节 RAG，复用 SiliconFlow 兼容 OpenAI Embeddings 接口）=====
   9 | EMBEDDING_BASE_URL=https://api.siliconflow.cn/v1
  10 | EMBEDDING_API_KEY=sk-xxxx
  11 | EMBEDDING_MODEL=BAAI/bge-m3

  data/terminologies.txt（3

### 2.2 项目结构一览（第 26 节新增部分）

```
smartbrief/backend/
├── rag/                        #  ★ 新增整个包：RAG 知识库（复刻 RAG 课程包的工程封装）
│   ├── loaders.py              #    Document 模型 + 多格式加载（txt/md/docx/pdf/xlsx）
│   ├── chunkers.py             #    RecursiveChunker：递归分块（512 字 + 64 重叠）
│   ├── embeddings.py           #    SiliconFlow bge-m3 嵌入封装（从 .env 读配置）
│   ├── vectorstores.py         #    ChromaDB 封装：md5 稳定 ID / 幂等去重 / 相似检索
│   └── store.py                #  ★ 门面：rag_collection + memory_collection 双集合管理
├── memory/                     #  ★ 新增整个包：记忆
│   ├── checkpointer.py         #    短期记忆：LangGraph 快照（backend/checkpoints.db）
│   └── long_term_memory.py     #    长期记忆：成功报告入库 + 相似检索（memory_collection）
├── data/
│   ├── terminologies.txt       #  ★ 预置 30 条行业术语（每行 术语：定义）
│   ├── templates/              #  ★ 预置 5 份报告模板（template_*.md）
│   └── chroma_db/              #  ★ ChromaDB 持久化目录（向量数据）
├── agents/
│   ├── planner.py              #  ★ 生成 Plan 前先查长期记忆（Few-shot + memory_hits）
│   └── react_writer.py         #  ★ 撰写前先查 RAG 术语表（注入 Prompt + 日志标注）
├── graph/
│   ├── state.py                #  ★ 白板新增第 6 组字段：memory_hits
│   └── workflow.py             #  ★ compile 挂 Checkpointer + 任务成功后 remember
├── main.py                     #  ★ 启动：RAG 预置 seed + 遗留 running 任务标 paused
├── checkpoints.db              #  ★ 短期记忆快照库（LangGraph 框架自动写）
└── api/routes.py               #  ★ /knowledge/*、/memory/*、/tasks/{id}/resume、DELETE

frontend/src/views/
├── KnowledgeBase.vue           # ★ 新增：知识库页（文档管理 + RAG/记忆检索测试 + 记忆开关）
├── TaskDetail.vue              # ★ 增强：「长期记忆命中」卡片 + 「恢复执行」按钮
└── Tasks.vue                   # ★ 增强：状态「已暂停」+ 删除 / 恢复操作
```

> 读代码前先记住一句话：**第 26 节所有改动，都是为了让系统 ——**
> **① 需要什么知识才取什么（RAG 检索）；② 做到一半能接着做（短期记忆）；③ 做过的下次直接复用（长期记忆）**。


---
## Part 3：先立框架 —— 智能体的"记忆"分几种？
---

动手前，先把"记忆"这个词拆清楚。结合 Part 1 的三个痛点，我们把智能体的记忆分成三类：

| 记忆类型 | 类比人类 | 存在哪 | 生命周期 | 解决哪个反馈 |
|---------|---------|--------|---------|-------------|
| **语义知识（RAG）** | 工具书 / 词典，用的时候查 | ChromaDB `rag_collection` | 可增删（预置 + 上传） | 反馈一：用词不专业 |
| **短期记忆** | 手头任务干到一半的状态 | `checkpoints.db` 图快照 | 随任务走，可断点续跑 | 反馈二：重启白干 |
| **长期记忆** | 上次项目的经验 | ChromaDB `memory_collection` | 跨任务长期保留 | 反馈三：没有积累 |

### 3.1 别忘了：第 24 节起就有的"第四种存储"

| 存储 | 谁在写 | 记录什么 | 写入时机 |
|------|--------|---------|---------|
| `data.db`（业务库） | 业务代码 `task_repository` | 任务的**结果**：清单 / Plan / 日志 / 报告 | 图跑完后**显式**写一次 |
| `checkpoints.db`（26 节） | LangGraph 框架（Checkpointer） | 任务的**进行时**：图执行到哪个节点、白板当前值 | 每执行完一个节点**自动**写 |
| `chroma_db/`（26 节） | RAG / 记忆模块 | 系统的**知识**：术语 / 模板 / 上传文档 + 历史报告 | 预置 seed / 上传 / 任务成功时 |

> 一句话分工：**`data.db` 记"交差的结果"，`checkpoints.db` 记"干到哪了"，`chroma_db` 记"会用的知识和经验"**。
> 三者互不干扰 —— 删任务只清 `data.db` 的业务行，绝不动 `checkpoints.db`（那是框架的地盘）。

### 3.2 为什么长期记忆和 RAG 都用向量库，却要**两个集合**？

先抛结论，后面 Part 6.3 展开：`rag_collection` 存**外部资料**（术语/模板/上传文档，写给 Writer 用）；
`memory_collection` 存**我们自己的产出**（成功任务报告，写给 Planner 用）。两个集合的
**写入者、查询时机、用途语义完全不同**，硬塞一个集合会让"查术语"和"查经验"互相污染。


In [4]:
# ============================================================
# Cell 3：在本机找到"几类存储"的真实文件，建立体感
# ============================================================
from pathlib import Path

data_dir = Path(BACKEND) / "data"
candidates = [
    ("业务库 data.db   （任务的'结果'，第 24 节起）", Path(BACKEND) / "data.db"),
    ("图快照 checkpoints.db（短期记忆，26 节新增）", Path(BACKEND) / "checkpoints.db"),
    ("向量库 chroma_db/ （知识 + 长期记忆，26 节新增）", data_dir / "chroma_db"),
]
print("=" * 55)
print("  三类存储在本机的真实文件")
print("=" * 55)
for name, p in candidates:
    exists = p.exists()
    extra = ""
    if p.is_file():
        extra = f"  ({p.stat().st_size / 1024:.1f} KB)"
    elif p.is_dir():
        extra = "  (目录)"
    print(f"  [{'有' if exists else '无'}] {name}{extra}")
print()
print("  说明：")
print("    - 后面的 Cell 16 演示会把这三个文件全部指向临时目录，绝不污染本机数据")
print("    - checkpoints.db 你平时不需要手动维护 —— LangGraph 框架自动写入")


  三类存储在本机的真实文件
  [有] 业务库 data.db   （任务的'结果'，第 24 节起）  (744.0 KB)
  [有] 图快照 checkpoints.db（短期记忆，26 节新增）  (876.0 KB)
  [有] 向量库 chroma_db/ （知识 + 长期记忆，26 节新增）  (目录)

  说明：
    - 后面的 Cell 16 演示会把这三个文件全部指向临时目录，绝不污染本机数据
    - checkpoints.db 你平时不需要手动维护 —— LangGraph 框架自动写入


---
## Part 4：给 Writer 一本"专业词典" —— RAG 术语注入
---

### 4.1 从痛点出发：Writer 为什么会"用词外行"？

复盘第 24/25 节的 Writer：它的输入只有 `研究子任务 + 研究数据摘要`，然后就让 LLM 写。
问题在于：

| # | 症状 | 根因 |
|---|------|------|
| 1 | "同比增长"写成"同月比"、"渗透率"说成"普及速度" | 没有**领域术语知识**可查 |
| 2 | 同一指标每份报告叫法不一（YoY / 同比 / 年增…） | 没有**统一口径**约束 |
| 3 | 行业有 30 条沉淀术语、5 份模板却用不上 | 资料在文件里，但**不在 Prompt 里** |

**设计取向**（先想清楚，再写代码）：

> 不给 Writer 塞 30 条术语全文，而是让它**"动笔前先检索"** —— 拿当前撰写任务去向量库
> 语义检索最相关的 2~3 条术语定义，注入 Prompt。术语越多越要"检索"，而不是"搬运"。

### 4.2 RAG 原理：一句结论 + 回看指引（不重复讲）

> **结论**：RAG = **先"入库"再"检索增强"**。入库：把资料切块 → 嵌入成向量 → 存向量库；
> 检索增强：把问题嵌入成向量 → 按余弦相似度召回 Top-k 文本块 → **作为参考拼进 Prompt** → LLM 再作答。

| RAG 环节 | 本节用的实现 | 原理在 10-15 节哪个文件讲过 |
|---------|-------------|------------------------------|
| 多格式加载 | `rag/loaders.py` | `01-多格式文档加载与数据清洗.ipynb` |
| 文本分块 | `rag/chunkers.py` | `02-文本分块与向量嵌入.ipynb` |
| 嵌入模型 | `rag/embeddings.py`（bge-m3） | `02-文本分块与向量嵌入.ipynb` |
| 向量库/检索 | `rag/vectorstores.py`（ChromaDB） | `03-向量数据库与语义检索.ipynb` |
| 检索增强问答 | Writer 注入（本节重点） | `04-检索增强生成与精准问答.ipynb` |
| 评测优化 | （本项目以日志/单测做轻量验证） | `05-RAG系统评测、优化与部署.ipynb` |

> 所以 Part 4 的代码走读会**很快**：原理不重复，重点看"项目在复刻时改进了什么"、
> "Writer 是怎么消费检索结果的"。

### 4.3 先看门面 `rag/store.py`：双集合从哪来

`rag/store.py` 是整个 RAG 子系统的"门面"（Facade），外部只跟它打交道。
先记住三个常量与两个集合的定位，再看 Cell 4 的代码：


In [5]:
# ============================================================
# Cell 4：走读 rag/store.py —— 门面与预置 seed
# ============================================================
import inspect

try:
    from rag import store as rag_store

    print("=" * 55)
    print("  两个向量集合 + 数据目录常量")
    print("=" * 55)
    print(f"  RAG_COLLECTION     = {rag_store.RAG_COLLECTION!r}")
    print(f"  MEMORY_COLLECTION  = {rag_store.MEMORY_COLLECTION!r}")
    print(f"  CHROMA_PERSIST_DIR = {rag_store.CHROMA_PERSIST_DIR}")
    print(f"  预置：模板 {rag_store.EXPECTED_TEMPLATES} 份、术语 {rag_store.EXPECTED_TERMS} 条")
    print()

    print("=" * 55)
    print("  seed_rag_knowledge 源码（预置数据，幂等 seed）")
    print("=" * 55)
    print(inspect.getsource(rag_store.seed_rag_knowledge))

except Exception as e:
    print("  [导入失败]", e)


  两个向量集合 + 数据目录常量
  RAG_COLLECTION     = 'rag_collection'
  MEMORY_COLLECTION  = 'memory_collection'
  CHROMA_PERSIST_DIR = C:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj05\smartbrief\backend\data\chroma_db
  预置：模板 5 份、术语 30 条

  seed_rag_knowledge 源码（预置数据，幂等 seed）
def seed_rag_knowledge(persist_dir: Optional[str] = None,
                       templates_dir: Optional[Path] = None,
                       terms_file: Optional[Path] = None) -> dict:
    """预置知识库数据（幂等，已就绪则跳过；首次/不足时补齐并入库）。

    供 main.py 生命周期启动时调用一次；测试可通过显式传参隔离。
    :return: {"templates": n, "terms": m, "skipped": bool}
    """
    try:
        store = get_store(persist_dir, RAG_COLLECTION)
        t_count = store.count({"category": "template"})
        g_count = store.count({"category": "terminology"})
        if t_count >= EXPECTED_TEMPLATES and g_count >= EXPECTED_TERMS:
            return {"templates": t_count, "terms": g_count, "skipped": True}
        # 补齐缺失部分：逐类别统计现有 source 集，只入库尚未预置的
        seed_doc

In [6]:
# ============================================================
# Cell 5：继续走读 store.py —— 上传入库管线 & Writer 专用检索
# ============================================================
try:
    from rag import store as rag_store

    print("=" * 55)
    print("  ingest_file 源码（上传文件 → 入库：存盘→解析→分块→向量化）")
    print("=" * 55)
    print(inspect.getsource(rag_store.ingest_file))

    print()
    print("=" * 55)
    print("  search_rag_terms 源码（Writer 专用：默认取 terminology 类，阈值 0.35）")
    print("=" * 55)
    print(inspect.getsource(rag_store.search_rag_terms))

except Exception as e:
    print("  [导入失败]", e)


  ingest_file 源码（上传文件 → 入库：存盘→解析→分块→向量化）
def ingest_file(file_bytes: bytes, filename: str, category: str = "upload",
                persist_dir: Optional[str] = None) -> dict:
    """把上传文件写入知识库：存盘 → 分格式加载 → 递归分块 → 入库。

    :return: {"source": filename, "added": chunk数, "chunks": n}
    """
    upload_dir = Path(persist_dir and str(Path(persist_dir).parent / "uploads") or UPLOADS_DIR)
    upload_dir.mkdir(parents=True, exist_ok=True)
    target = upload_dir / filename
    target.write_bytes(file_bytes)

    docs = DocumentLoader().load(str(target))
    chunks = RecursiveChunker().chunk(docs)
    for chunk in chunks:
        chunk.metadata["category"] = category
        chunk.metadata["source"] = filename          # 以文件名作为用户可读 source（覆盖绝对路径）
        chunk.metadata["uploaded_name"] = filename
    store = get_store(persist_dir, RAG_COLLECTION)
    added = store.add_documents(chunks)
    return {"source": filename, "category": category, "added": len(added), "chunks": len(chunks)}


  searc

### 4.4 支撑引擎：加载 / 分块 / 向量库 —— 复刻 RAG 课程包时改进了什么？

`ingest_file` 里三步（`DocumentLoader` → `RecursiveChunker` → `ChromaVectorStore`）
是直接复刻第 10-15 节 RAG 课程包（`zhiqida-rag`）的工程封装。**原理不重复**，
我们只对比"复刻时动了哪些地方、为什么动"：

| 组件 | RAG 课程包 | SmartBrief 复刻版 | 为什么改 |
|------|-----------|------------------|---------|
| 配置来源 | 课程包自己的 `ZHIQIDA_*` 配置 | 读 SmartBrief `backend/.env` 的 `EMBEDDING_*` | 一个项目一套配置，不引入第二套配置体系 |
| 文档 ID | 内置 `hash()`（进程级随机盐） | `md5(content)` **稳定哈希** | 💡 见下方"一个真实的 Bug 故事" |
| 数据目录 | 课程包 settings | 由 `store.py` 显式传 `persist_dir` | 测试可用临时目录隔离，不污染真实库 |
| 召回过滤 | 课程包较简单 | 支持 `where`（category/type）+ 阈值 | 双集合、术语注入都需要分类过滤 |

> 💡 **一个真实的 Bug 故事（为什么用 md5 做 ID）**：
> Python 内置 `hash("同一段文字")` 在**每次进程启动时结果都不同**（字符串哈希带随机盐，
> 防哈希碰撞攻击）。RAG 课程包早期用 `hash()` 生成文档 ID —— 只要后端重启一次，
> 相同内容会算出**新的 ID**，于是重复入库、越积越多，无法幂等重建。
> 第 26 节改为 `md5(content)`：**同一段文字在任何进程、任何时候 ID 都一致**，
> `add_documents` 里"先查已存在 ID → 跳过"天然去重，重启、重复上传都不会产生垃圾块。
> 这个小改动，是"把课程 demo 变成生产代码"的典型差异。

Cell 6 / Cell 7 快速走读这三个引擎文件（重点看差异点，其余一笔带过）：


In [7]:
# ============================================================
# Cell 6：走读 rag/loaders.py 与 rag/chunkers.py（关键部分）
# ============================================================
try:
    from rag.loaders import DocumentLoader
    from rag.chunkers import RecursiveChunker

    print("=" * 55)
    print("  loaders.DocumentLoader（策略分发：按扩展名选 Loader）")
    print("=" * 55)
    print(inspect.getsource(DocumentLoader.load))

    print("=" * 55)
    print("  chunkers.RecursiveChunker（递归分块核心）")
    print("=" * 55)
    print(inspect.getsource(RecursiveChunker._recursive_split))

except Exception as e:
    print("  [导入失败]", e)


  loaders.DocumentLoader（策略分发：按扩展名选 Loader）
    def load(self, file_path: str) -> List[Document]:
        ext = os.path.splitext(file_path)[1].lower()
        loader = self.loaders.get(ext)
        if loader is None:
            raise ValueError(f"不支持的文件类型：{ext}")
        return loader.load(file_path)

  chunkers.RecursiveChunker（递归分块核心）
    def _recursive_split(self, text: str) -> List[str]:
        """递归切分文本。

        流程：文本不超长直接返回 → 找到第一个出现的分隔符切分（贪心合并小块）
        → 对仍然过大的块递归（用更低优先级分隔符）→ 全失败时按字符兜底。
        """
        if len(text) <= self.chunk_size:
            return [text]

        chunks = []
        for sep in self.separators:
            if sep == "":
                # 兜底：按字符强制切分（带重叠）
                chunks = self._split_by_char(text)
                break
            if sep in text:
                segments = text.split(sep)
                current_chunk = ""
                for segment in segments:
                    candidate = current_chunk + sep + segment if current_chunk e

In [8]:
# ============================================================
# Cell 7：走读 rag/vectorstores.py —— md5 稳定 ID 与幂等入库
# ============================================================
try:
    from rag.vectorstores import ChromaVectorStore

    print("=" * 55)
    print("  _doc_id（md5 稳定 ID —— 4.4 那个 Bug 故事的答案）")
    print("=" * 55)
    print(inspect.getsource(ChromaVectorStore._doc_id))

    print()
    print("=" * 55)
    print("  add_documents（先过滤已存在 ID，再批量 upsert）")
    print("=" * 55)
    src = inspect.getsource(ChromaVectorStore.add_documents)
    print(src)

except Exception as e:
    print("  [导入失败]", e)


  _doc_id（md5 稳定 ID —— 4.4 那个 Bug 故事的答案）
    def _doc_id(self, content: str) -> str:
        """稳定 ID：对文本内容做 md5（跨进程一致，天然去重 + 支持幂等重建）。"""
        return hashlib.md5(content.encode("utf-8")).hexdigest()


  add_documents（先过滤已存在 ID，再批量 upsert）
    def add_documents(self, documents: List[Document],
                      embeddings: Optional[List[List[float]]] = None) -> List[str]:
        """添加文档：生成稳定 ID → 自动嵌入 → 过滤已存在 → 批量 upsert。

        :return: 实际新增的 doc id 列表
        """
        self._lazy_init()
        if embeddings is None:
            embeddings = self.embeddings.embed_documents([d.page_content for d in documents])

        ids, texts, metadatas = [], [], []
        for i, doc in enumerate(documents):
            ids.append(self._doc_id(doc.page_content))
            texts.append(doc.page_content)
            metadatas.append(doc.metadata)

        # 去重：跳过已存在的 ID（幂等，重复上传不产生垃圾块）
        existing = set(self._collection.get(ids=ids).get("ids", []))
        new_ids, new_emb, new_tex

### 4.5 Writer 怎么"用"这本词典？—— 检索 → 日志 → 注入

有了 `rag/store.py`，第 26 节对 `react_writer.py` 的改动非常克制（只加三件事）：

1. **写前检索**：拿"当前撰写子任务"去 `search_rag_terms`（默认 k=3、相似度阈值 0.35，
   且只查 `terminology` 类术语）—— 不是全文搬运，是"语义最相关的 3 条"；
2. **留痕**：无论有没有命中，都往 `react_logs` 追加一条 `Thought` 日志：
   - 命中 → `RAG 检索到 3 条相关术语：YoY：同比增长率…；渗透率：…`
   - 没命中 → `RAG 检索：未找到相关术语定义`
   前端任务详情页的日志流，据此就能看到"Writer 到底查没查、查到了什么"；
3. **注入**：命中的术语定义按 `- 术语：定义` 追加到 user Prompt 末尾，
   同时 Action 日志标注 `llm_gen(...)（已注入 RAG 术语 2 条）`。

> 🛡️ **容错语义**：检索是**增强**而非**依赖** —— 术语库为空 / Key 缺失 / 网络异常，
> `_retrieve_terms` 一律吞掉返回 `[]`，Writer 照常撰写（降级为"无词典"路径），
> 主流程永不因 RAG 子系统故障而中断。这也呼应了第 25 节的"降级可见"原则。

Cell 8 走读最终代码：


In [9]:
# ============================================================
# Cell 8：走读 agents/react_writer.py —— RAG 术语注入（第 26 节新增段）
# ============================================================
try:
    from agents import react_writer

    print("=" * 55)
    print("  _retrieve_terms（线程池包装检索，便于测试 patch）")
    print("=" * 55)
    print(inspect.getsource(react_writer._retrieve_terms))

    print()
    print("=" * 55)
    print("  react_writer_run（重点看第 26 节新增的 3 处：检索/日志/注入）")
    print("=" * 55)
    print(inspect.getsource(react_writer.react_writer_run))

except Exception as e:
    print("  [导入失败]", e)


  _retrieve_terms（线程池包装检索，便于测试 patch）
async def _retrieve_terms(task: str) -> list:
    """（第 26 节）检索与撰写任务相关的 RAG 术语定义（线程池包装，便于测试 patch）。

    :return: 术语定义文本列表；RAG 不可用时返回 []（不影响撰写）
    """
    try:
        return await asyncio.to_thread(rag_store.search_rag_terms, task)
    except Exception:
        return []


  react_writer_run（重点看第 26 节新增的 3 处：检索/日志/注入）
async def react_writer_run(task: str, step_id: int, logs: list, research_data: dict) -> str:
    """对单个 writing 步骤执行 ReAct 循环（行动 = llm_gen）。

    与 Researcher 的区别：Writer 的"行动"不是调用外部搜索工具，
    而是直接调用 LLM 生成章节文本（llm_gen），随后把生成结果作为 Observation。

    执行步骤：
    1. 记 Thought：开始撰写 + Action：llm_gen
    2. 把研究数据压缩成摘要，拼进 user prompt
    3. 调 LLM 生成章节 Markdown
    4. LLM 失败 → 返回占位文本（不中断整个任务）
    5. 记 Observation：已生成章节草稿

    :param task: 该步骤的撰写子任务
    :param step_id: 对应 plan 步骤 id
    :param logs: 共享 react_logs 列表（原地追加）
    :param research_data: 已收集的研究数据（供引用）
    :return: 生成的章节 Markdown 文本
    """
    # ---------- 第 26 节：撰写前检索 RAG 术语表 ---------

---
## Part 5：短期记忆 —— 让任务"断电续跑"
---

### 5.1 痛点具体化：第 24/25 节的任务为什么"怕重启"？

回看 `workflow.run_task()`：它一次性 `await graph.ainvoke(initial)`，**图跑完才把结果写进 `data.db`**。
于是有个巨大的空窗：

```
planner → scheduler(研究) → scheduler(撰写) → reviewer → rewriter → reviewer → ... → 落库
    │                                  ↑
    └──── 图跑了一半，后端被重启 / 部署 / 崩溃 ────┘
            内存里的白板全没了，任务卡在 running，再也动不了
```

更麻烦的是：图里还有第 25 节的**审核回路**，"从头重跑"会重复烧 LLM 的钱，
而且重跑结果可能和上次不一样（LLM 非确定性）—— 客户会投诉"为什么两次跑得不一样"。

### 5.2 方案：LangGraph Checkpointer（短期记忆）

LangGraph 内置了记忆机制 —— **Checkpointer（检查点）**：

| 概念 | 说明 |
|------|------|
| Checkpoint | 每执行完一个节点，框架自动把"白板 state + 图执行到哪"存一份快照 |
| thread_id | 一条"会话/任务"的标识。本项目让 `thread_id = task_id`（一一对应） |
| compile(checkpointer=...) | 编译图时挂载检查点器，快照才会被写入 |
| 续跑 | `graph.ainvoke(None, {"configurable": {"thread_id": task_id}})` —— 不传新输入，从最近快照继续 |

本项目把 Checkpointer 持久化到 SQLite（`backend/checkpoints.db`，`langgraph-checkpoint-sqlite`），
这样**后端进程重启后，快照依然在磁盘上**，这就是"断电续跑"的物理基础。

### 5.3 三个"库"的分工又来了（重要）

| 文件 | 谁写 | 写什么 | 什么时候 |
|------|------|--------|---------|
| `data.db` | 我们的业务代码 | 任务**结果**（status/plan/report/日志） | 图跑完后写一次 |
| `checkpoints.db` | **LangGraph 框架** | 图**运行时状态**（节点进度 + 白板值） | 每个节点执行后自动写 |

> 为什么不能让业务代码去"自己存断点"（Part 1 方案 C）？
> 因为 checkpoint 需要记录的是**图引擎内部状态**（当前在哪个超步、缓存了哪些写入），
> 这是框架的地盘。我们只需把 `task_id` 作为 `thread_id`，让两条线通过 task_id 对齐。
> 这样即使后端重启，任务详情页依然能查到业务结果，也能用快照把"进行时"续上。

Cell 9 先到本机真实看一眼 `checkpoints.db` 里到底存了什么：


In [10]:
# ============================================================
# Cell 9：解剖本机 checkpoints.db —— 短期记忆的"物理形态"
# ============================================================
import sqlite3

db_file = os.path.join(BACKEND, "checkpoints.db")
print("=" * 55)
print(f"  checkpoints.db 内容（{db_file}）")
print("=" * 55)

if not os.path.exists(db_file):
    print("  文件尚不存在 —— 没关系，后面的端到端演示会生成它（在临时目录）")
else:
    con = sqlite3.connect(db_file)
    tables = [r[0] for r in con.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    for t in tables:
        try:
            count = con.execute(f"SELECT COUNT(*) FROM '{t}'").fetchone()[0]
        except Exception:
            count = "?"
        print(f"  表 {t:<22} 行数: {count}")
    con.close()
    print()
    print("  说明：这些表由 langgraph-checkpoint-sqlite 自动维护：")
    print("    - checkpoints:     每个超步一张状态快照（含 thread_id / checkpoint_id）")
    print("    - checkpoint_writes: 节点返回的增量（pending writes）")
    print("    你不需要（也不应该）手写 SQL 操作它们 —— 框架全包了")


  checkpoints.db 内容（c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj05\smartbrief\backend\checkpoints.db）
  表 checkpoints            行数: 16
  表 writes                 行数: 96

  说明：这些表由 langgraph-checkpoint-sqlite 自动维护：
    - checkpoints:     每个超步一张状态快照（含 thread_id / checkpoint_id）
    - checkpoint_writes: 节点返回的增量（pending writes）
    你不需要（也不应该）手写 SQL 操作它们 —— 框架全包了


### 5.4 看封装 `memory/checkpointer.py` 前，先懂三个 API 细节

| 细节 | 说明 |
|------|------|
| ① `from_conn_string()` 的语义变了 | v3.x 的 `AsyncSqliteSaver.from_conn_string()` 返回的不是 Saver 实例，而是**连接生命周期管理器** —— 必须 `async with open_checkpointer() as (cp, kind):` 把 `compile + invoke` 包在连接生命周期里，节点快照才会真正落盘 |
| ② 只读查询走同步 | `has_checkpoint()` / `list_snapshots()` 用**同步版** `SqliteSaver` 短连接查询，避免在 async 上下文里维护长连接 |
| ③ 缺依赖自动降级 | `langgraph-checkpoint-sqlite` 未装 / 文件打开失败 → 降级为进程内 `MemorySaver`（单例复用），保证同一进程内仍能 resume（重启即失效，属于"尽力而为"） |

> 这些细节不是炫技 —— 前两条来自**真实库版本升级踩坑**（README/CLAUDE.md 里有记录），
> 第三条来自第 25 节就立下的原则：**任何外部依赖都不能让主流程死掉**。

Cell 10 走读封装代码：


In [11]:
# ============================================================
# Cell 10：走读 memory/checkpointer.py —— 短期记忆封装
# ============================================================
try:
    from memory import checkpointer as cp_mod

    print("=" * 55)
    print("  open_checkpointer（async with 连接生命周期 + 降级）")
    print("=" * 55)
    print(inspect.getsource(cp_mod.open_checkpointer))

    print()
    print("=" * 55)
    print("  has_checkpoint（同步只读：任务是否有可续跑快照）")
    print("=" * 55)
    print(inspect.getsource(cp_mod.has_checkpoint))

    print()
    print("=" * 55)
    print("  list_snapshots（同步只读：列出断点列表，供前端展示）")
    print("=" * 55)
    print(inspect.getsource(cp_mod.list_snapshots))

except Exception as e:
    print("  [导入失败]", e)


  open_checkpointer（async with 连接生命周期 + 降级）
@asynccontextmanager
async def open_checkpointer(
    db_path: Optional[str] = None,
) -> AsyncIterator[Tuple[Any, str]]:
    """在连接生命周期内打开持久化 Checkpointer。

    新版 langgraph-checkpoint-sqlite 要求 `async with from_conn_string(...)`
    进入连接生命周期：期间图每次节点执行后自动向 SQLite 落快照，退出即关闭连接。
    sqlite 依赖/文件不可用则降级为进程内 MemorySaver（kind="memory"）。

    :param db_path: 覆盖默认 backend/checkpoints.db（测试隔离用）
    :yield: (checkpointer, kind)，kind ∈ {"sqlite", "memory"}
    """
    path = Path(db_path or CHECKPOINT_DB)
    cp: Any = None
    kind = "memory"
    opener: Any = None
    try:
        from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

        # 显式打开连接（真正连库发生在 opener.__aenter__，返回 saver 实例）；
        # 失败即降级。注意：降级分支绝不能包住 yield —— body 抛出的异常若被
        # except 吞掉并再次 yield，asynccontextmanager 会报
        # "generator didn't stop after athrow()"。
        opener = AsyncSqliteSaver.from_conn_string(str(path))
        cp = await opener.__aenter__()
    

### 5.5 两个藏在注释里的工程教训（读代码时注意）

1. **降级分支绝不能包住 `yield`**（`checkpointer.py` 里专门有注释）：
   `asynccontextmanager` 中，如果 `try` 里先 `yield`、又在外层 `except` 里再 `yield` 一次，
   当 body 抛异常被吞掉后，框架会报 `generator didn't stop after athrow()`。
   所以项目写法是：**先把 Saver 准备好（失败就换成 MemorySaver），再进第二个 `try: yield`**。
   —— 这是"降级代码怎么写得优雅"的真实教材。

2. **快照元数据用 `*rest` 兼容展开**：不同版本 `list()` 返回的 `CheckpointTuple`
   字段数不同，`for config, _checkpoint, meta, *_rest in configs` 保证升级不炸。
   —— 依赖第三方框架时，对外部数据结构要"留一手"。

### 5.6 还没完：谁来把 running 变成 paused？—— 预告 Part 8

光有快照还不够：**后端重启后，那个卡住的 running 任务需要被识别出来**。
项目在 `main.py` 启动时扫描所有 running 任务 → 标记为 `paused`（已暂停）；
前端看到 paused 才展示「恢复执行」按钮，点击后调 `POST /tasks/{id}/resume` → `run_task(resume=True)`。
这条完整的状态机链路放到 Part 8 一起看。


---
## Part 6：长期记忆 —— 把成功经验"存得下、找得到"
---

### 6.1 痛点：同主题第二次，还是从零开始

第 5 节解决了"做到一半重启"，但还有第三个痛点：**跨任务的积累**。

> 三个月前做完『智能手表』的报告，Plan 是：市场规模 → 竞争格局 → 技术趋势 → 撰写；
> 这次做『智能手表 2026 趋势』，Planner 又是从零开始自由发挥，可能排成
> 技术拆解 → 政策 → 竞品 → 撰写 —— 结构漂移、每次都不一样。

> 真实团队怎么解决？**老带新**：新人写方案前，先翻出"上次那个相似项目的方案"当参考。
> 系统同样可以：Planner 拆 Plan 前，先去"经验库"里找一份**最相似的历史报告**，
> 把它的**章节结构**作为 Few-shot 参考注入 Prompt —— 只借结构，不背正文。

### 6.2 长期记忆"存什么"？—— 不是存全文，是存"可复用的骨架"

`memory/long_term_memory.py` 的设计（先看设计再看代码）：

| 设计点 | 取值 | 为什么 |
|--------|------|--------|
| 存的内容 | `主题：{topic}\n\n{报告全文}` | 向量检索 + 后续抽结构都靠它 |
| 入库长度上限 | `_REPORT_LIMIT = 6000` 字符 | bge-m3 有 8192 token 上限，超长截断保安全 |
| 元数据 | `type=report` + `topic/task_id/created_at` | 检索时可 `where={"type":"report"}` 精确过滤 |
| ID | md5(全文) 稳定哈希 | 与 RAG 同一个去重套路：同一任务重复 remember 不产生垃圾 |
| 总开关 | `MEMORY_ENABLED`（.env 或 `/memory/toggle`） | **逻辑开关**：关闭即短路，已存数据不清空（Part 9 展开） |
| 写入时机 | **任务成功后**（workflow 调 `_remember_task`） | 只把"验证过的产出"当经验，失败任务不配进记忆 |

`recall()` 返回的结构（供 Planner 消费）：

```python
[{"topic": "智能手表 2025 市场",          # 历史报告主题
  "structure": ["## 市场规模与增速", ...],  # 从报告中抽出的章节标题
  "snippet": "主题：…（前 200 字）",
  "score": 0.912,                        # 相似度
  "task_id": "task_xxxx", "created_at": "…"}]
```

Cell 11 走读最终代码：


In [12]:
# ============================================================
# Cell 11：走读 memory/long_term_memory.py —— 长期记忆
# ============================================================
try:
    from memory import long_term_memory as ltm

    print("=" * 55)
    print("  remember（任务成功后写入记忆库）")
    print("=" * 55)
    print(inspect.getsource(ltm.remember))

    print()
    print("=" * 55)
    print("  _extract_structure（从报告抽章节标题，Few-shot 用）")
    print("=" * 55)
    print(inspect.getsource(ltm._extract_structure))

    print()
    print("=" * 55)
    print("  recall（按主题检索相似历史报告）")
    print("=" * 55)
    print(inspect.getsource(ltm.recall))

except Exception as e:
    print("  [导入失败]", e)


  remember（任务成功后写入记忆库）
def remember(topic: str, final_report: str, task_id: Optional[str] = None,
             persist_dir: Optional[str] = None) -> bool:
    """把一次成功任务存入长期记忆（幂等：md5 稳定 ID 去重）。

    :return: True 表示新写入；False 表示重复（或失败/为空跳过）
    """
    if not _MEMORY_ENABLED:
        return False
    report = (final_report or "").strip()
    if not report:
        return False
    try:
        store = get_store(persist_dir, MEMORY_COLLECTION)
        from rag.loaders import Document

        doc = Document(
            page_content=f"主题：{topic}\n\n{report[:_REPORT_LIMIT]}",
            metadata={
                "type": "report",
                "topic": topic,
                "task_id": task_id or "",
                "created_at": _now(),
            },
        )
        added = store.add_documents([doc])
        if added:
            logger.info("长期记忆已写入：topic=%s task_id=%s", topic, task_id)
        return bool(added)
    except Exception as exc:  # pragma: no cover - 真实异常路径（网络/依赖）
  

### 6.3 为什么长期记忆 ≠ RAG 知识库？—— 两个集合的分工

都是 ChromaDB，为什么 `rag_collection` 和 `memory_collection` 必须分开？

| 维度 | `rag_collection`（知识库） | `memory_collection`（长期记忆） |
|------|---------------------------|-------------------------------|
| 内容 | 术语 / 模板 / 用户上传文档（**外部资料**） | 我们自己跑成功的 (topic, final_report)（**内部产出**） |
| 写者 | 预置 seed + 用户上传 | `workflow._remember_task`（任务成功时） |
| 读者 | Writer 撰写时查术语（`search_rag_terms`） | Planner 规划时查经验（`recall`）+ `/memory/search` |
| 过滤字段 | `category=template/terminology/upload` | `type=report` |
| 语义 | "写的时候该怎么用词" | "规划的时候该按什么框架" |

> 两者文档形态也不同：术语一条只有几十字、模板有固定结构；报告是几千字的全文。
> 混在一个集合里，检索时 `where` 过滤会很绕，且"术语命中"和"经验命中"会互相串味。
> 拆开 = 关注点分离：**知识库管"用词"，记忆库管"框架"**。

> 🛡️ **容错**：`remember/recall` 内部任何异常都被吞掉并记 warning —— 记忆系统挂了，
> 任务照常执行，只是"这次没记住/没查到"。


---
## Part 7：Planner 的经验复用 —— 相似报告做 Few-shot
---

### 7.1 思路：只借"结构"，不背"全文"

Planner 拿到 topic 后，原本直接 `SYSTEM_PROMPT + 研究主题` 让 LLM 出 Plan。
第 26 节在中间插了一步 **recall（查经验）**：

```
topic ──► recall(topic) ── 找到最相似的历史报告
              │
              ▼
    _build_few_shot(top) ──► "【长期记忆参考】历史主题：智能手表 2025 市场
                              章节结构：- ## 市场规模与增速
                                        - ## 竞争格局
                                        - ..."
              │
              ▼
      user = "研究主题：{topic}" + few_shot   ──► LLM 出 Plan
```

| 设计点 | 取值 | 为什么 |
|--------|------|--------|
| 只取 Top-1 命中 | `hits[0]` | 参考要"精"不要"杂"，一份结构足够对齐框架 |
| 只注入章节结构 | `_build_few_shot` 取 `structure[:10]` | 避免把几千字正文塞进 Prompt（Part 1 方案 B 的病根） |
| 无命中时不注入 | structure 为空 → 返回空串 | 全新主题：没有历史参考时走原路径，不硬凑 |
| 命中要"留痕" | 写进 `state.memory_hits` + 日志 `检索到历史报告…` | 前端"记忆命中"卡片 + 用户可解释性 |

### 7.2 白板新增第 6 组字段：memory_hits

`graph/state.py` 新增（对应前端 TaskDetail「长期记忆命中」卡片的数据源）：

```python
memory_hits: List[Dict]   # [{"topic","score","task_id","created_at"}]
```

注意记忆命中里**故意不存 structure 全文**，只存 topic / score / task_id / created_at ——
卡片只需要"命中了谁、多像"；结构详情由前端按需调 `/memory/search` 再取。
另外 Planner 里用 `task_id` 做**续跑去重**：`run_task(resume=True)` 重进 planner_node 时，
state 已带旧命中，不能再重复追加同一条（看 Cell 12 里的 `any(m.get("task_id")...)` 判断）。

Cell 12 / Cell 13 走读最终代码：


In [13]:
# ============================================================
# Cell 12：走读 agents/planner.py —— 记忆增强后的 Planner
# ============================================================
try:
    from agents import planner

    print("=" * 55)
    print("  _build_few_shot（把 1 条历史命中构造成 Few-shot 文本）")
    print("=" * 55)
    print(inspect.getsource(planner._build_few_shot))

    print()
    print("=" * 55)
    print("  _recall_memory（线程池包装，测试可 patch 隔离）")
    print("=" * 55)
    print(inspect.getsource(planner._recall_memory))

    print()
    print("=" * 55)
    print("  planner_node（第 26 节新增段在函数中部）")
    print("=" * 55)
    print(inspect.getsource(planner.planner_node))

except Exception as e:
    print("  [导入失败]", e)


  _build_few_shot（把 1 条历史命中构造成 Few-shot 文本）
def _build_few_shot(hit: Dict) -> str:
    """把 1 条历史报告命中构造成 Few-shot 参考文本（章节结构优先）。

    :param hit: recall 返回的 {"topic", "structure", "snippet", "score", ...}
    :return: 注入 user prompt 的参考段落；无有效结构时返回空串
    """
    structure = hit.get("structure") or []
    if not structure:
        return ""
    lines = "\n".join(f"- {h}" for h in structure[:10])
    return (
        "\n\n【长期记忆参考】以下是一次已完成的历史相似报告的结构，"
        "请参考其章节组织方式安排本次计划的 researcher/writer 步骤：\n"
        f"历史主题：{hit.get('topic', '')}\n{lines}"
    )


  _recall_memory（线程池包装，测试可 patch 隔离）
async def _recall_memory(topic: str) -> List[Dict]:
    """长期记忆检索（线程池包装同步 ChromaDB 调用，便于测试 patch 隔离）。"""
    try:
        return await asyncio.to_thread(long_term_memory.recall, topic)
    except Exception as exc:                      # 记忆子系统异常不阻断规划
        logger.warning("Planner 检索长期记忆失败：%s", exc)
        return []


  planner_node（第 26 节新增段在函数中部）
async def planner_node(state: dict) -> dict:
    """

In [14]:
# ============================================================
# Cell 13：看白板新增的第 6 组字段（state.py + 初始状态）
# ============================================================
try:
    from graph.state import AgentState
    from graph import workflow

    print("=" * 55)
    print("  AgentState 类源码（第 6 组：长期记忆命中，见文件末尾）")
    print("=" * 55)
    src = inspect.getsource(AgentState)
    lines = src.splitlines()
    # 只打印第 6 组注释块 + memory_hits 字段（结构骨架）
    start = next(i for i, l in enumerate(lines) if "6. 长期记忆命中" in l)
    for line in lines[start:]:
        print(line)
    print()

    print("=" * 55)
    print("  workflow._initial_state（memory_hits 的初始值）")
    print("=" * 55)
    print(inspect.getsource(workflow._initial_state))

except Exception as e:
    print("  [导入失败]", e)


  AgentState 类源码（第 6 组：长期记忆命中，见文件末尾）
    6. 长期记忆命中（第 26 节）：memory_hits
    """
    # ---------- 1. 任务信息 ----------
    task_id: str              # 任务唯一 id（由 API 层生成，形如 task_ab12cd34）
    topic: str                # 用户提交的研究主题（如 "智能手表 2025 市场"）

    # ---------- 2. Plan 进度 ----------
    plan: List[Dict]                # 步骤列表：[{"id":1, "agent":"researcher", "task":"...", "depends_on":[]}]
                                    # agent 取值：researcher（联网研究）/ writer（撰写章节）
    current_step_index: int         # 当前进度下标（= 已完成步骤数，供前端 Plan 看板定位）
    completed_steps: List[int]      # 已完成步骤 id 列表（如 [1, 2]）

    # ---------- 3. ReAct 日志 ----------
    react_logs: List[Dict]          # 日志流：[{"type":"Thought|Action|Observation", "content":"...", "step_id":1}]
                                    # 前端任务详情页按 step_id 分组展示，就是从这里来的

    # ---------- 4. 产出数据 ----------
    research_data: Dict             # 研究数据：{"step_1": {"observations": [...]}, "step_2": {...}}
    draft_content: str              # 报告草稿（writer

---
## Part 8：把记忆接进图 —— workflow.py / main.py 的改动
---

### 8.1 一张图看懂第 26 节在 workflow 层的 4 处改动

```
START → planner（★ 先查长期记忆 → Few-shot + memory_hits）
         │
         ▼
       scheduler（Researcher / ★ Writer 先查 RAG 术语表）
         │
         ▼
       reviewer ─passed→（★ run_task 内 remember 成功经验）→ END
         │
     不通过 → rewriter → 回 reviewer
         │
         └ 每个节点执行后：Checkpointer 自动落快照到 checkpoints.db（★）
```

| # | 改动 | 代码位置 |
|---|------|---------|
| ① | `build_graph(checkpointer=...)`：编译时挂载 Checkpointer（thread_id=task_id） | `build_graph` / `run_task` |
| ② | 续跑语义：`resume=True` 且有快照 → `graph.ainvoke(None, config)` 从断点继续 | `run_task` |
| ③ | 任务成功后 `_remember_task` → `long_term_memory.remember`（容错线程池） | `_remember_task` |
| ④ | `memory_hits` 随最终状态一起落库（tasks 表新列） | `run_task` 的 `update_task` |

Cell 14 走读最终代码：


In [15]:
# ============================================================
# Cell 14：走读 graph/workflow.py —— 记忆怎么接进图
# ============================================================
try:
    from graph import workflow

    print("=" * 55)
    print("  _remember_task（任务成功后写长期记忆，容错不阻断）")
    print("=" * 55)
    print(inspect.getsource(workflow._remember_task))

    print()
    print("=" * 55)
    print("  run_task（Checkpointer 连接生命周期 + resume 语义 + memory_hits 落库）")
    print("=" * 55)
    print(inspect.getsource(workflow.run_task))

    print()
    print("=" * 55)
    print("  build_graph（compile(checkpointer=...) —— 挂载短期记忆）")
    print("=" * 55)
    print(inspect.getsource(workflow.build_graph))

except Exception as e:
    print("  [导入失败]", e)


  _remember_task（任务成功后写长期记忆，容错不阻断）
async def _remember_task(task_id: str, topic: str, final_report: str) -> None:
    """（第 26 节）任务成功后把 (topic, final_report) 写入长期记忆（容错，不阻断流程）。

    长期记忆由 ChromaDB 承担：embedding 走网络，故放进线程池；
    任何失败仅记 warning，绝不影响任务状态回写。
    """
    try:
        await asyncio.to_thread(long_term_memory.remember, topic, final_report, task_id)
    except Exception as exc:  # pragma: no cover - 真实异常路径
        logger.warning("任务 %s 写入长期记忆失败（忽略）：%s", task_id, exc)


  run_task（Checkpointer 连接生命周期 + resume 语义 + memory_hits 落库）
async def run_task(task_id: str, topic: str, resume: bool = False) -> dict:
    """执行完整流程：运行图 → 落库日志与最终状态。

    流程示意：
        1. 构造初始 state（白板）；resume=True 且有快照时改为从断点继续
        2. graph.ainvoke(initial / None)：LangGraph 自动按边执行
           planner（生成计划）→ scheduler（派发 ReAct 执行）
           → reviewer（审核草稿；A2A 不通过则返工给 rewriter 覆盖式重写，最多返工 2 次）
        3. 把图内产生的 react_logs 批量落库（task_logs 表）
        4. 把最终状态写回 tasks 表（status=completed），
           含 A2A 审核结果字段：re

### 8.2 服务重启的完整状态机（paused → resume）

有快照还不够，还要把"卡死的任务"暴露给用户操作。完整链路：

```
后端运行中，任务 X 卡在 scheduler（内存里跑到一半）
        │
        ★ 后端被强制关闭 / 崩溃 / 部署重启
        ▼
main.py lifespan 启动扫描：
   "SELECT * FROM tasks WHERE status='running'"  → 全部标为 paused（已暂停）
        │
        ▼
前端任务列表/详情：状态=「已暂停」→ 显示「恢复执行」按钮
        │  点击
        ▼
POST /api/v1/tasks/{task_id}/resume
   → task_repository.update_task(status=running)
   → asyncio.create_task(workflow.run_task(task_id, topic, resume=True))
        │
        ▼
run_task(resume=True)：有 checkpoint 快照 → graph.ainvoke(None) 从最近节点续跑
                         无快照（比如刚建任务还没跑图）→ 自动从头执行兜底
```

> 注意边界：**completed 的任务不允许 resume**（返回 400）；paused / failed 才允许。
> 这是因为"已完成 = 没有可续的进行时"。

Cell 15 走读 `main.py` 的启动兜底与 API 的 resume 端点：


In [16]:
# ============================================================
# Cell 15：走读 main.py（启动兜底）与 routes.resume_task
# ============================================================
import inspect as _inspect

print("=" * 55)
print("  main.py（lifespan：遗留 running → paused + RAG 预置 seed）")
print("=" * 55)
main_path = os.path.join(BACKEND, "main.py")
src_lines = open(main_path, encoding="utf-8").read().splitlines()
# 只打印 lifespan 关键段（init_db 到 skills 初始化之前）
for no, line in enumerate(src_lines, start=1):
    if 22 <= no <= 58:
        print(f"{no:>4} | {line}")
print()

try:
    from api import routes
    print("=" * 55)
    print("  routes.resume_task（断点续跑端点）")
    print("=" * 55)
    print(_inspect.getsource(routes.resume_task))
except Exception as e:
    print("  [导入失败] 这是 API 层（连带加载技能池/MCP 等模块）：", e)
    print("  接口逻辑已在 main.py / routes.py 源码中说明，不影响后续学习。")


  main.py（lifespan：遗留 running → paused + RAG 预置 seed）
  22 | async def lifespan(app: FastAPI):
  23 |     """应用生命周期：启动时初始化各子系统，关闭时回收 MCP 连接。
  24 | 
  25 |     启动顺序（任何一步失败都会阻止应用启动，便于尽早暴露配置问题）：
  26 |     业务库建表 → RAG 知识库预置 → 内置工具注册表 → 技能池 → 拉起启用的 MCP Server。
  27 | 
  28 |     第 26 节 RAG 预置：把 data/templates/（5 份模板）与 data/terminologies.txt
  29 |     （30 个术语）写入 ChromaDB rag_collection（幂等：已就绪自动跳过）。
  30 |     测试/CI 可通过环境变量 SMARTBRIEF_RAG_SEED=0 关闭（避免离线环境网络调用）。
  31 |     """
  32 |     # 初始化 SQLite 业务库（tasks / task_logs 表）
  33 |     database.init_db()
  34 |     # 意外中断兜底（paused 状态）：上次进程退出遗留的 running 任务统一置为 paused，
  35 |     # 用户可点「恢复执行」从 SqliteSaver(checkpoints.db) 断点续跑（第 26 节短期记忆）
  36 |     try:
  37 |         interrupted = task_repository.mark_interrupted_tasks_as_paused()
  38 |         if interrupted:
  39 |             logger.warning("发现 %d 个因服务中断遗留的任务，已标记为 paused（可恢复执行）", interrupted)
  40 |     except Exception as exc:  # pragma: no cover - 真实异常路径
  41 |         logger.warning("

---
## Part 9：API 与前端闭环 —— 让"知识"和"记忆"看得见、摸得着
---

### 9.1 第 26 节新增 / 增强的接口

| 方法 | 路径 | 功能 | 关键点 |
|------|------|------|--------|
| POST | `/api/v1/knowledge/documents` | 上传文档入知识库 | 分类：template / terminology / upload |
| GET | `/api/v1/knowledge/documents` | 文档列表 | 按 (category, source) **聚合**成一个文件条目 |
| GET | `/api/v1/knowledge/document` | 文档详情（全部 chunk） | ChromaDB `$and` 双条件过滤 |
| DELETE | `/api/v1/knowledge/document` | 删除文档 | 文件级删除，不残留孤儿 chunk |
| POST | `/api/v1/knowledge/search` | RAG 语义检索测试 | 与 Writer 术语注入同一条链路 |
| GET | `/api/v1/memory/search` | 检索长期记忆 | Planner `recall()` 的同一条链路 |
| GET | `/api/v1/memory/status` | 开关状态 + 报告数 | 关闭时计数归 0（不查库） |
| POST | `/api/v1/memory/toggle` | 长期记忆逻辑开关 | 关闭不清空，重开即恢复 |
| POST | `/api/v1/tasks/{id}/resume` | 恢复 paused/failed 任务 | completed 返回 400 |
| DELETE | `/api/v1/tasks/{id}` | 删除任务 | 级联删除 ReAct 日志 |

> 三个值得品的设计点：
> 1. **文档列表要"聚合"**：ChromaDB 里一份文件被切成很多 chunk 分条存储，直接返回会刷屏；
>    `store.list_documents` 按 `(category, source)` 聚合，前端看到的是"一份文件"，点「查看」再展开 chunk —— 这是"存储模型 vs 展示模型"的典型差异。
> 2. **删除是文件级不是 chunk 级**：`delete_document` 先 `$and` 查出该文件全部 chunk id 再批量删，保证不留孤儿块。
> 3. **上传走 `asyncio.to_thread`**：文档解析 / 分块 / 向量化都是阻塞 IO + 网络调用，
>    丢进线程池避免卡住 FastAPI 事件循环 —— 与第 25 节"同步慢操作让路"同一个套路。

### 9.2 前端两处主要变化

- **`KnowledgeBase.vue`（路由 /knowledge，新页面）**：分类统计卡（模板 / 术语 / 文档数量 +
  长期记忆开关与报告数）、文档列表（分类筛选：全部 / 模板 / 术语 / 文档）、上传（选分类）、
  「查看」chunk 弹窗（含元数据）、「删除」、「RAG 检索测试」与「长期记忆检索」。
- **`TaskDetail.vue` / `Tasks.vue`（增强）**：
  - 「长期记忆命中」卡片：展示 Planner 命中历史报告的 topic / 相似度 / 结构标签 / snippet；
  - 状态增加「已暂停」；仅 paused 状态展示「恢复执行」按钮；列表新增「删除」操作（确认弹窗）。

Cell 16 走读 API 层的知识库与记忆端点：


In [17]:
# ============================================================
# Cell 16：走读 api/routes.py —— 知识库 + 记忆端点
# ============================================================
try:
    from api import routes

    for fn_name in ("upload_knowledge_document", "search_knowledge",
                    "memory_status", "memory_toggle"):
        fn = getattr(routes, fn_name)
        print("=" * 55)
        print(f"  routes.{fn_name}")
        print("=" * 55)
        print(inspect.getsource(fn))
        print()

except Exception as e:
    print("  [导入失败] 这是 API 层（连带加载技能池/MCP 等模块）：", e)
    print("  接口代码已在上方 MD 讲解，可直接跳到 Part 10 演示。")


  routes.upload_knowledge_document
@router.post("/knowledge/documents")
async def upload_knowledge_document(
    file: UploadFile = File(...),
    category: str = Form("upload", description="分类：upload（默认）/ template / terminology"),
):
    """上传文档到 RAG 知识库（教学链路：存盘 → 分格式解析 → 递归分块 → 向量化入库）。

    与 zhiqida-rag 一致的处理管线（rag_store.ingest_file 内部）：
      1. 原始字节先落盘到 data/uploads/（保留源文件，便于溯源与重试）
      2. DocumentLoader 按扩展名分发：txt/md 读文本、docx 提段落、
         pdf 每页一个 Document、xlsx 每 sheet 一个 Document
      3. RecursiveChunker 按段落/句子边界切块（512 字 + 64 字重叠）
      4. ChromaVectorStore 用 bge-m3 向量化 + md5 稳定 ID 去重后入库
    上传后立即可在知识库列表/检索测试里看到效果（无需重启后端）。
    """
    content = await file.read()
    if not content:
        raise HTTPException(status_code=400, detail="上传文件为空")
    filename = file.filename or "unnamed.txt"
    try:
        result = await asyncio.to_thread(
            rag_store.ingest_file, content, filename, category or "upload"
        )
        return result
    except Exception as exc:
    

---
## Part 10：端到端演示 —— 把"知识 + 记忆"的闭环完整跑一遍
---

把前面所有零件拼起来：**真实 LangGraph 图 + 模拟的 LLM / 搜索 / 向量检索 / 长期记忆 + 临时目录**，
完整走一遍：

1. **任务 A**「智能手表 2025 市场」：系统**没有任何经验** —— Planner 查记忆为空，直接规划；
   Writer 撰写时**检索到 2 条 RAG 术语**并注入 Prompt；任务成功后把报告写入长期记忆；
   图每走一个节点都自动落 Checkpointer 快照。
2. **任务 B**「智能手表 2026 趋势展望」：Planner 查记忆**命中任务 A** —— 拿到它的章节结构做
   Few-shot；`memory_hits` 随任务落库。

> 💡 演示全程**不联网、不花钱**：RAG 检索与长期记忆都用内存版"替身"（真实环境由
> ChromaDB + bge-m3 承担），LLM 用 `unittest.mock` 替身。真实运行时只需去掉 patch，
> 配上 `backend/.env` 的 Key 即可 —— 逻辑完全一致。

Cell 17（较长）执行演示：


In [18]:
# ============================================================
# Cell 17：端到端演示 —— 记忆闭环（模拟 LLM / 检索 / 记忆）
# ============================================================
import asyncio
import json
import os
import tempfile
from datetime import datetime
from unittest.mock import AsyncMock, patch

print("=" * 55)
print("  Step 0：数据文件全部指向临时目录（不污染本机数据）")
print("=" * 55)
tmpdir = tempfile.mkdtemp(prefix="sb26_")

# 1) 业务库 -> 临时
from db import database
from db import task_repository
database.init_db(os.path.join(tmpdir, "data.db"))

# 2) Checkpointer 库 -> 临时（LangGraph 框架写快照）
from memory import checkpointer as cp_mod
cp_mod.CHECKPOINT_DB = os.path.join(tmpdir, "checkpoints.db")
cp_mod.reset_checkpointer()

print("  临时目录:", tmpdir)
print()

# ---- 假响应对象：项目代码统一访问 resp.choices[0].message.content ----
class _Msg:
    def __init__(self, content):
        self.content = content
        self.tool_calls = None

class _Choice:
    def __init__(self, msg):
        self.message = msg

class _Resp:
    def __init__(self, text):
        self.choices = [_Choice(_Msg(text))]

# ---- "长期记忆"内存替身：模拟 Chroma 的 remember / recall ----
MEM = []          # [{"topic", "report", "task_id", "created_at"}]

def _now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def _structure(text):
    return [l.strip() for l in (text or "").splitlines()
            if l.strip().startswith("#")][:12]

def fake_remember(topic, final_report, task_id=None, persist_dir=None):
    text = (final_report or "").strip()
    if not text:
        return False
    MEM.append({"topic": topic, "report": text,
                "task_id": task_id, "created_at": _now()})
    print(f"     [长期记忆写入] task={task_id} topic={topic} "
          f"章节数={len(_structure(text))}")
    return True

def fake_recall(topic, k=3, persist_dir=None):
    # 极简"语义相似"：存在长度>=4 的公共子串即视为相近
    # （真实环境由 bge-m3 向量 + 余弦相似度决定，这里只做演示）
    def similar(a, b):
        a, b = str(a), str(b)
        return any(a[i:i + 4] in b for i in range(max(1, len(a) - 3)))

    hits = []
    for m in reversed(MEM):                # 最近完成的优先
        if similar(topic, m["topic"]):
            hits.append({
                "topic": m["topic"],
                "structure": _structure(m["report"]),
                "snippet": m["report"][:200],
                "score": round(0.95 - 0.1 * len(hits), 4),
                "task_id": m["task_id"],
                "created_at": m["created_at"],
            })
            if len(hits) >= k:
                break
    return hits

# ---- RAG 术语表"内存替身"：模拟 search_rag_terms 命中 ----
FAKE_TERMS = [
    "YoY：同比增长率（Year-over-Year），指本期与上年同期相比的增减幅度",
    "渗透率：某产品在目标市场中的普及程度，用于判断行业所处成长阶段",
]

# ---- 假 LLM：按"调用者"分别返回 ----
_searched = set()         # 已搜过的研究子任务（第二次问 -> done）
_planner_prompts = []     # 收集 Planner 收到的 user 文本，用来观察 Few-shot

def _mk_plan(topic):
    return json.dumps([
        {"id": 1, "agent": "researcher",
         "task": f"搜索 {topic} 的市场规模与增速", "depends_on": []},
        {"id": 2, "agent": "researcher",
         "task": f"搜索 {topic} 的头部厂商与竞争格局", "depends_on": [1]},
        {"id": 3, "agent": "writer",
         "task": f"撰写《{topic}》市场调研报告", "depends_on": [1, 2]},
    ], ensure_ascii=False)

async def fake_planner_chat(messages, **kwargs):
    user = messages[-1]["content"]        # 形如："研究主题：X" / "研究主题：X【长期记忆参考】…"
    topic = user.split("研究主题：", 1)[1].split("【长期记忆参考】", 1)[0].strip()
    _planner_prompts.append(user)
    return _Resp(_mk_plan(topic))

async def fake_researcher_chat(messages, **kwargs):
    task = messages[-1]["content"].split("研究子任务：", 1)[-1].strip()
    if task in _searched:                  # 同一子任务已搜过一轮 -> 收工
        return _Resp('{"action": "done"}')
    _searched.add(task)
    return _Resp(json.dumps({"action": "web_search", "query": task},
                            ensure_ascii=False))

async def fake_writer_chat(messages, **kwargs):
    return _Resp(
        "# 智能手表市场调研报告\n\n"
        "## 市场规模与增速\n\n2025 年全球智能手表市场规模约 300 亿美元，"
        "同比增长约 15%（YoY，来源：示例调研机构）。\n\n"
        "## 竞争格局\n\n头部厂商：Apple、华为、三星、小米（来源：示例）。\n\n"
        "## 渗透率与前景\n\n行业渗透率持续提升，仍处快速成长期。")

async def fake_reviewer_chat(messages, **kwargs):
    return _Resp('{"passed": true, "comments": []}')   # 一次通过（本课重点不在审核）

def fake_search(name, arguments):
    q = (arguments or {}).get("query", "")
    if "厂商" in q or "竞争" in q:
        return {"content": "Apple、华为、三星、小米占据前四（来源：示例调研）",
                "source": "mock"}
    return {"content": "2025 年全球智能手表市场规模约 300 亿美元，同比 +15%（来源：示例调研）",
            "source": "mock"}

# ---- 组装并运行一次完整任务 ----
async def run_demo(task_id, topic, tag):
    print("=" * 55)
    print(f"  {tag}  task={task_id}  topic={topic}")
    print("=" * 55)
    _searched.clear()
    task_repository.create_task(task_id, topic)

    with (
        patch.object(planner_mod, "chat", fake_planner_chat),
        patch.object(researcher_mod, "chat", fake_researcher_chat),
        patch.object(researcher_mod.mcp_manager, "call_tool",
                     AsyncMock(side_effect=fake_search)),
        patch.object(writer_mod, "chat", fake_writer_chat),
        patch.object(reviewer_mod, "chat", fake_reviewer_chat),
        patch.object(rewriter_mod, "chat", fake_reviewer_chat),
        patch.object(writer_mod.rag_store, "search_rag_terms",
                     lambda task, **kw: FAKE_TERMS),           # RAG 术语命中
        patch.object(ltm_mod, "remember", fake_remember),       # 长期记忆：写
        patch.object(ltm_mod, "recall", fake_recall),           # 长期记忆：查
    ):
        final = await workflow.run_task(task_id, topic)

    logs = final.get("react_logs", [])
    print(f"  [完成] 日志 {len(logs)} 条 | memory_hits {len(final.get('memory_hits', []))} 条")
    print("  - Planner 阶段（step 0）日志：")
    for lg in logs:
        if lg.get("step_id") == 0 and lg.get("type") in ("Thought", "Observation"):
            print(f"      [{lg['type']}] {lg['content'][:80]}")
    print("  - Writer 阶段 RAG 术语日志：")
    for lg in logs:
        if "RAG" in lg.get("content", ""):
            print(f"      [step{lg.get('step_id')}] {lg['content'][:110]}")
    if final.get("memory_hits"):
        print("  - memory_hits（落库的长期记忆命中）：")
        for h in final["memory_hits"]:
            print(f"      topic={h['topic']}  score={h['score']}  "
                  f"task_id={h['task_id']}")
    has_cp = cp_mod.has_checkpoint(task_id)
    snaps = cp_mod.list_snapshots(task_id)
    print(f"  - 短期记忆快照：has_checkpoint={has_cp}，"
          f"共 {len(snaps)} 个（每执行一个节点自动存 1 档）")
    print()
    return final

# ---- 引用真实模块（patch 的靶子）----
from agents import planner as planner_mod
from agents import react_researcher as researcher_mod
from agents import react_writer as writer_mod
from agents import reviewer as reviewer_mod
from agents import rewriter as rewriter_mod
from memory import long_term_memory as ltm_mod
from graph import workflow

print("【任务 A】首次做『智能手表』—— 系统还没有任何长期记忆")
final_a = await run_demo("task_mem_01", "智能手表 2025 市场", "任务 A")

print("【任务 B】再次做相似的『智能手表 2026 趋势』—— 应该能'想起'任务 A")
final_b = await run_demo("task_mem_02", "智能手表 2026 趋势展望", "任务 B")

# ---- 验证：任务 B 的 Planner 是否真的收到了 Few-shot ----
print("=" * 55)
print("  验证：任务 B 的 Planner Prompt 是否包含【长期记忆参考】")
print("=" * 55)
prompt_b = [u for u in _planner_prompts if "2026" in u]
if prompt_b and "【长期记忆参考】" in prompt_b[0]:
    seg = prompt_b[0].split("【长期记忆参考】", 1)[1].strip()
    print("  命中！任务 B 参考了任务 A 的报告结构：")
    for line in seg.splitlines():
        print("    " + line)
else:
    print("  未注入（MEM 为空或相似度判否）")

print()
print("  两次任务结束后，长期记忆里共", len(MEM), "条经验：")
for m in MEM:
    print(f"    - {m['topic']}  (task={m['task_id']})")
    for h in _structure(m["report"]):
        print(f"        {h}")


SQLite Checkpointer 不可用（No module named 'langgraph.checkpoint.sqlite'），降级为进程内 MemorySaver


  Step 0：数据文件全部指向临时目录（不污染本机数据）
  临时目录: C:\Users\13691\AppData\Local\Temp\sb26_k7jbh8y4

【任务 A】首次做『智能手表』—— 系统还没有任何长期记忆
  任务 A  task=task_mem_01  topic=智能手表 2025 市场


SQLite 查询任务 task_mem_01 快照失败（No module named 'langgraph.checkpoint.sqlite'），退回进程内 MemorySaver
SQLite 读取任务 task_mem_01 快照失败（No module named 'langgraph.checkpoint.sqlite'），退回进程内 MemorySaver
SQLite Checkpointer 不可用（No module named 'langgraph.checkpoint.sqlite'），降级为进程内 MemorySaver


     [长期记忆写入] task=task_mem_01 topic=智能手表 2025 市场 章节数=4
  [完成] 日志 30 条 | memory_hits 0 条
  - Planner 阶段（step 0）日志：
      [Thought] 已生成执行计划：3 个步骤（researcher, researcher, writer）
      [Thought] 开始审核草稿（第 1 次审核）
      [Observation] 第 1 次审核通过
  - Writer 阶段 RAG 术语日志：
      [step3] RAG 检索到 2 条相关术语：YoY：同比增长率（Year-over-Year），指本期与上年同期相比的增减幅度；渗透率：某产品在目标市场中的普及程度，用于判断行业所处成长阶段
      [step3] llm_gen(query=撰写本章节草稿)（已注入 RAG 术语 2 条）
  - 短期记忆快照：has_checkpoint=True，共 5 个（每执行一个节点自动存 1 档）

【任务 B】再次做相似的『智能手表 2026 趋势』—— 应该能'想起'任务 A
  任务 B  task=task_mem_02  topic=智能手表 2026 趋势展望


SQLite 查询任务 task_mem_02 快照失败（No module named 'langgraph.checkpoint.sqlite'），退回进程内 MemorySaver
SQLite 读取任务 task_mem_02 快照失败（No module named 'langgraph.checkpoint.sqlite'），退回进程内 MemorySaver


     [长期记忆写入] task=task_mem_02 topic=智能手表 2026 趋势展望 章节数=4
  [完成] 日志 31 条 | memory_hits 1 条
  - Planner 阶段（step 0）日志：
      [Thought] 已生成执行计划：3 个步骤（researcher, researcher, writer）
      [Thought] 检索到历史报告：智能手表 2025 市场，结构类似，已作为 Few-shot 参考
      [Thought] 开始审核草稿（第 1 次审核）
      [Observation] 第 1 次审核通过
  - Writer 阶段 RAG 术语日志：
      [step3] RAG 检索到 2 条相关术语：YoY：同比增长率（Year-over-Year），指本期与上年同期相比的增减幅度；渗透率：某产品在目标市场中的普及程度，用于判断行业所处成长阶段
      [step3] llm_gen(query=撰写本章节草稿)（已注入 RAG 术语 2 条）
  - memory_hits（落库的长期记忆命中）：
      topic=智能手表 2025 市场  score=0.95  task_id=task_mem_01
  - 短期记忆快照：has_checkpoint=True，共 5 个（每执行一个节点自动存 1 档）

  验证：任务 B 的 Planner Prompt 是否包含【长期记忆参考】
  命中！任务 B 参考了任务 A 的报告结构：
    以下是一次已完成的历史相似报告的结构，请参考其章节组织方式安排本次计划的 researcher/writer 步骤：
    历史主题：智能手表 2025 市场
    - # 智能手表市场调研报告
    - ## 市场规模与增速
    - ## 竞争格局
    - ## 渗透率与前景

  两次任务结束后，长期记忆里共 2 条经验：
    - 智能手表 2025 市场  (task=task_mem_01)
        # 智能手表市场调研报告
        ## 市场规模与增速
        ## 竞争格局
        ## 渗透率与前景
    - 智能手表 2026 趋势展望  (tas

---

### 10.1 看懂上面的输出

你会在输出里看到一条完整的"记忆流水账"，对照下表理解每一段：

| 你看到的输出 | 它证明什么 | 对应 Part |
|------------|-----------|----------|
| 任务 A 的 Planner 阶段**只有**"已生成执行计划" | 首次运行无历史 → 走**无记忆路径** | Part 7 |
| 任务 A/B 的 Writer 阶段出现 `RAG 检索到 2 条相关术语：YoY…；渗透率…` 且 Action 标注"已注入" | 写手动笔前**真的查了术语库**并注入 Prompt | Part 4 |
| 任务 A 结束时打印 `[长期记忆写入] … 章节数=3` | 任务成功后经验被**自动记住** | Part 6 |
| 任务 B 的 Planner 阶段出现"检索到历史报告…已作为 Few-shot 参考"、末尾 `memory_hits` 有 1 条 | 规划前**想起了**上次的相似报告 | Part 7 |
| 验证段打印出任务 A 的三级章节结构 | Few-shot 内容 = **结构骨架**，不是全文 | Part 7 |
| 每次任务后 `has_checkpoint=True、共 N 个快照` | 图每走一个节点都自动落档（短期记忆） | Part 5 |

> 换个玩法：
> 1. 把 `FAKE_TERMS` 置为空列表再跑 —— Writer 日志会变成
>    `RAG 检索：未找到相关术语定义`，证明"降级为无词典路径"仍然能写完报告；
> 2. 把任务 A 的主题改成完全不相关的（如"分布式光伏"）再跑任务 B —— `similar` 判否，
>    Planner 回到无记忆路径 —— 感受"相似度"才是记忆闸门（真实系统用向量相似度）。


---
## Part 11：课程总结
---

### 我们经历了什么？

| 演进阶段 | 遇到的问题 | 解决方案 |
|---------|-----------|---------|
| 知识 | Writer 用词外行、口径不一 | RAG 术语表：撰写前检索 Top-k 定义注入 Prompt |
| 检索 | 术语会越来越多，全文塞不进 | 向量语义检索（md5 稳定 ID / 分类过滤 / 阈值） |
| 短期记忆 | 图跑一半重启就白干 | LangGraph Checkpointer → `checkpoints.db`，每节点自动快照 |
| 状态机 | 重启后卡死的 running 没人管 | 启动扫描 → paused；`/tasks/{id}/resume` 断点续跑 |
| 长期记忆 | 同主题第二次仍从零开始 | 成功报告入库 → `recall` 相似检索 |
| 复用 | 历史报告全文太长没法参考 | Planner Few-shot：只借**章节结构** |
| 留痕 | 用户不知道系统"想没想起来" | `memory_hits` 落库 + 前端"长期记忆命中"卡片 |
| 降级 | RAG / 记忆服务挂了怎么办 | 检索失败一律返回空，主流程照跑、照写、照记日志 |

### 关键设计原则

1. **按需检索，而非全文搬运**：Prompt 里只放"此刻最相关"的 2~3 条知识；
2. **记忆三分**：语义知识（词典）/ 短期记忆（进行时）/ 长期记忆（经验），三种库分工明确、互不污染；
3. **框架的活不自己造轮子**：图状态快照交给 Checkpointer，业务代码只管"结果落库"；
4. **稳定 ID 是幂等的前提**：md5(content) 让重复入库/重复 remember 天然去重；
5. **记忆是增强不是依赖**：任何记忆/检索子系统故障，主流程都要能降级跑完；
6. **一切留痕**：RAG 命没命中、记忆想没想起，都要有日志、有字段、有前端展示。

### 下节课预告

**第 27 节：全栈评测与云端部署**
- 基于第 26 节的"知识 + 记忆"，搭一套自动化评测体系（10 个标准用例）
- 评测报告自动生成（JSON + Markdown + 雷达图）
- 打磨 5 个核心页面，前后端最终联调与部署

---

## 课后练习

### 练习一：你是架构师 —— 给长期记忆加"用户隔离"

现在的 `memory_collection` 是所有用户共享的。产品经理说："不同客户（如 A 客户 / B 客户）
的经验要隔离，A 客户做『便携储能』的经验不能污染 B 客户。"

请先思考再动手：
1. `remember()` / `recall()` 的 metadata 与 `where` 过滤要怎么改（提示：加 `user_id` 元数据 +
   `where={"$and": [{"type": "report"}, {"user_id": ...}]}`）？
2. Planner 调用 `_recall_memory` 时，`user_id` 从哪来？（API 层 → workflow → planner 的参数传递）
3. 跟 6.3"两个集合分开"的道理做对比：这次是在**同一个集合里**做隔离，还是拆集合？

### 练习二：动手调参 —— 术语注入的"多与准"

打开 `rag/store.py` 的 `search_rag_terms`：
1. 把 `k` 从 3 改成 5、把 `threshold` 从 0.35 改成 0.5，各重跑一遍 Cell 17 的演示，
   观察 Writer 日志里"检索到 N 条"的变化；
2. 思考：`threshold` 设得过高会怎样？（查不到 → 降级为"无词典"路径）—— 这印证了哪条设计原则？

### 练习三：观察真实运行（需 API Key + 前后端）

1. 配置好 `backend/.env`（`OPENAI_*` 与 `EMBEDDING_*`），启动后端与前端；
2. 打开「知识库」页：能看到预置的 5 份模板 + 30 条术语；上传一份 .md 文档，点「查看」看它的 chunk；
3. 用「RAG 检索测试」输入"同比增长率 英文缩写是什么"，观察是否召回 `YoY` 术语；
4. 提交主题"便携储能"任务，在任务详情页观察：Writer 日志出现「RAG 检索到 N 条相关术语…」；
5. 再做一次相似主题（如"便携储能 2026"）→ 详情页出现「长期记忆命中」卡片；
6. 任务执行中**强杀后端进程**再重启 → 列表出现「已暂停」→ 点「恢复执行」，观察从断点继续。

---

## 参考资料

- 规格：`多智能体协作开发PRD（拆分）.md —— 子文档 5（第 26 节 RAG 知识库与智能体记忆深度融合）`
- RAG 原理（回顾用）：`资料/.../02-大模型全栈开发实战/10-15/slides/01~05`（加载/分块/向量库/检索增强/评测）
- 代码：`backend/rag/*` —— loaders / chunkers / embeddings / vectorstores / store
- 代码：`backend/memory/checkpointer.py` + `long_term_memory.py`
- 代码：`backend/agents/planner.py` / `react_writer.py`（记忆增强）
- 代码：`backend/graph/workflow.py` / `state.py`；`backend/main.py`；`backend/api/routes.py`
- 官方：LangGraph 持久化（Checkpointer）文档 https://langchain-ai.github.io/langgraph/concepts/persistence/

> 🎉 恭喜你完成了本节课的学习！
> 现在你的 SmartBrief 已经**会分工、会把关、还有了知识和记忆** ——
> 下一节（第 27 节）我们给它装上"考试系统"（自动化评测），然后走向部署。
